# CoT Iterative Refinement — Test Notebook

This notebook tests the `cot_n` iterative refinement feature and verifies that existing functionality (best-of-n, caching, step configs) is unaffected.

**Sections:**
1. Imports & setup
2. Unit tests — CoT helpers (no API key required)
3. Regression tests — StepConfig / AgentConfig serialization
4. Regression tests — existing best-of-n still works
5. Integration test — real LLM with cot_n=2 (requires OpenAI API key)

## 1. Imports & Setup

In [ ]:
import os
import sys
import difflib

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from Agent.config import StepConfig, AgentConfig
from Agent.data_agent import (
    SalesDataAgent,
    CoTRefinementLLM,
    _extract_step_output,
    _COT_SIMILARITY_THRESHOLD,
)

OPENAI_AVAILABLE = bool(os.getenv('OPENAI_API_KEY'))
print('All imports successful!')
print(f'OpenAI API key available: {OPENAI_AVAILABLE}')
print(f'CoT similarity threshold: {_COT_SIMILARITY_THRESHOLD}')

### Shared test helpers

In [ ]:
class FakeLLM:
    """Mock LLM: returns a fixed response and records all prompts received."""
    def __init__(self, response_text=''):
        self._response = response_text
        self.calls = []

    def invoke(self, prompt):
        self.calls.append(prompt)
        class _R:
            pass
        r = _R()
        r.content = self._response
        return r

    def __getattr__(self, name):
        return None


def make_core_fn(responses):
    """Returns a core_fn that steps through `responses` on each call."""
    idx = [0]
    call_log = []
    def core_fn(state, llm):
        sql = responses[min(idx[0], len(responses) - 1)]
        idx[0] += 1
        call_log.append(sql)
        return {'sql_query': sql}
    core_fn.call_log = call_log
    return core_fn


# Shared agent instance (no LLM calls during __init__)
agent = SalesDataAgent(model='gpt-4o-mini')
dummy_state = {'prompt': 'test'}
dummy_llm   = FakeLLM()

print('Helpers ready.')

## 2. Unit Tests — CoT Helpers (no API key required)

In [ ]:
print('=== Test 2.1: _extract_step_output ===')

assert _extract_step_output('lookup_sales_data', {'sql_query': 'SELECT 1'}) == 'SELECT 1'
assert _extract_step_output('decide_tool', {'tool_choice': 'analyzing_data'}) == 'analyzing_data'
assert _extract_step_output('analyzing_data', {'answer': ['first', 'second']}) == 'second'
assert _extract_step_output('create_visualization', {'answer': ['chart_code']}) == 'chart_code'
assert _extract_step_output('analyzing_data', {}) == ''  # empty fallback

print('  All assertions passed ✓')

In [ ]:
print('=== Test 2.2: CoTRefinementLLM prompt augmentation ===')

base = FakeLLM('SELECT 2 FROM sales')
wrapper = CoTRefinementLLM(base, previous_response='SELECT 1 FROM sales')
wrapper.invoke('Original prompt here')

received = base.calls[0]
assert 'Original prompt here' in received, 'Original prompt must be present'
assert 'ITERATIVE REFINEMENT'  in received, 'Refinement header must be present'
assert 'SELECT 1 FROM sales'   in received, 'Previous response must be quoted'

print('  Refinement suffix correctly appended ✓')
print(f'  Prompt grew from {len("Original prompt here")} to {len(received)} chars')

In [ ]:
print('=== Test 2.3: early stop on identical output ===')

# Iteration 2 returns same SQL → early stop; iteration 3 never executed
fn = make_core_fn(['SELECT SUM(a) FROM t', 'SELECT AVG(a) FROM t'])
initial = {'sql_query': 'SELECT SUM(a) FROM t'}

result = agent._apply_cot_iterations(
    step_name='lookup_sales_data', state=dummy_state,
    core_fn=fn, llm=dummy_llm, initial_result=initial, cot_n=3,
)

assert result['sql_query'] == 'SELECT SUM(a) FROM t', f"Unexpected: {result['sql_query']}"
assert len(fn.call_log) == 1, f'Expected 1 refinement call (early stop), got {len(fn.call_log)}'

print(f'  Early stop after 1 refinement call ✓  (3rd iteration not reached)')

In [ ]:
print('=== Test 2.4: all cot_n iterations run when outputs are dissimilar ===')
print(f'  similarity threshold = {_COT_SIMILARITY_THRESHOLD}')

# SELECT / UPDATE / DELETE are structurally incompatible → similarity << 0.95
s1 = 'SELECT SUM(revenue) FROM sales'
s2 = 'UPDATE inventory SET quantity = quantity - 1 WHERE product_id = 42'
s3 = 'DELETE FROM expired_sessions WHERE last_active < NOW() - INTERVAL 30 days'

r12 = difflib.SequenceMatcher(None, s1, s2).ratio()
r23 = difflib.SequenceMatcher(None, s2, s3).ratio()
print(f'  similarity(s1,s2)={r12:.3f}   similarity(s2,s3)={r23:.3f}')
assert r12 < _COT_SIMILARITY_THRESHOLD, f'Test strings too similar: {r12:.3f}'
assert r23 < _COT_SIMILARITY_THRESHOLD, f'Test strings too similar: {r23:.3f}'

fn2 = make_core_fn([s2, s3])
result2 = agent._apply_cot_iterations(
    step_name='lookup_sales_data', state=dummy_state,
    core_fn=fn2, llm=dummy_llm, initial_result={'sql_query': s1}, cot_n=3,
)

assert result2['sql_query'] == s3, f"Unexpected final SQL: {result2['sql_query']}"
assert len(fn2.call_log) == 2, f'Expected 2 refinement calls, got {len(fn2.call_log)}'

print(f'  All {len(fn2.call_log)} refinement iterations executed ✓')

In [ ]:
print('=== Test 2.5: cot_n=1 is a strict no-op ===')

fn3 = make_core_fn(['SHOULD NEVER BE CALLED'])
initial3 = {'sql_query': 'SELECT 1'}

result3 = agent._apply_cot_iterations(
    step_name='lookup_sales_data', state=dummy_state,
    core_fn=fn3, llm=dummy_llm, initial_result=initial3, cot_n=1,
)

assert result3 is initial3, 'cot_n=1 must return the exact same object'
assert len(fn3.call_log) == 0, 'cot_n=1 must make zero extra LLM calls'

print('  cot_n=1: initial result returned unchanged, zero extra calls ✓')
print('\n=== All unit tests passed ✓ ===')

## 3. Regression Tests — StepConfig / AgentConfig Serialization

In [ ]:
print('=== Regression 3.1: StepConfig default cot_n ===')

sc = StepConfig(n=3, temp_min=0.1, temp_max=0.9, step_name='test')
assert sc.cot_n == 1, f'Default cot_n should be 1, got {sc.cot_n}'
assert sc.n == 3
print(f'  cot_n default={sc.cot_n} ✓   existing fields: n={sc.n}, temp=[{sc.temp_min},{sc.temp_max}]')

print('\n=== Regression 3.2: StepConfig serialization round-trip ===')

sc2 = StepConfig(n=2, cot_n=4, temp_min=0.2, step_name='lookup_sales_data')
d = sc2.to_dict()
assert d['cot_n'] == 4,       f'cot_n not serialized correctly: {d}'
assert d['n'] == 2
assert 'eval_fn' not in d,       'eval_fn must not be serialized'
assert 'selection_fn' not in d,  'selection_fn must not be serialized'

sc3 = StepConfig.from_dict(d)
assert sc3.cot_n == 4
assert sc3.n == 2
print(f'  Serialized cot_n={d["cot_n"]} → deserialized cot_n={sc3.cot_n} ✓')

print('\n=== Regression 3.3: AgentConfig — all steps default to cot_n=1 ===')

ac = AgentConfig()
for step in ['decide_tool', 'lookup_sales_data', 'analyzing_data', 'create_visualization']:
    cfg = ac.get_step_config(step)
    assert cfg.cot_n == 1, f'{step} default cot_n should be 1, got {cfg.cot_n}'
    print(f'  {step}: cot_n={cfg.cot_n} ✓')

print('\n=== Regression 3.4: AgentConfig full serialization round-trip ===')

ac.analyzing_data.cot_n = 3
ac.lookup_sales_data.n  = 2
ac2 = AgentConfig.from_dict(ac.to_dict())
assert ac2.analyzing_data.cot_n  == 3
assert ac2.lookup_sales_data.n   == 2
assert ac2.decide_tool.cot_n     == 1  # unchanged
print(f'  analyzing_data.cot_n={ac2.analyzing_data.cot_n} ✓')
print(f'  lookup_sales_data.n={ac2.lookup_sales_data.n} ✓')
print(f'  decide_tool.cot_n={ac2.decide_tool.cot_n} (default, unchanged) ✓')

print('\nAll config regression tests passed ✓')

## 4. Regression Tests — Existing Best-of-n Unaffected

In [ ]:
print('=== Regression 4.1: cot_n=1 produces zero extra LLM calls ===')

extra_calls = []
def spy_fn(state, llm):
    extra_calls.append(1)
    return {'sql_query': 'SELECT 1'}

initial_obj = {'sql_query': 'SELECT 1'}
out = agent._apply_cot_iterations(
    step_name='lookup_sales_data', state=dummy_state,
    core_fn=spy_fn, llm=dummy_llm, initial_result=initial_obj, cot_n=1,
)

assert len(extra_calls) == 0, f'cot_n=1 must not call core_fn again, called {len(extra_calls)} times'
assert out is initial_obj,    'cot_n=1 must return the exact same result object'
print('  Zero extra calls, original object returned ✓')

print('\n=== Regression 4.2: get_temperatures() unaffected by cot_n ===')

sc_bon = StepConfig(n=3, temp_min=0.1, temp_max=0.9, cot_n=5)
temps = sc_bon.get_temperatures()
assert len(temps) == 3
assert abs(temps[0]  - 0.1) < 1e-9
assert abs(temps[-1] - 0.9) < 1e-9
print(f'  n=3, cot_n=5 → temps={[f"{t:.2f}" for t in temps]} ✓  (cot_n irrelevant here)')

print('\nAll best-of-n regression tests passed ✓')

## 5. Integration Test — Real LLM with cot_n=2

Requires `OPENAI_API_KEY`. Runs the full agent pipeline with `cot_n=2` and asserts that CoT iteration log lines appear.

In [ ]:
if OPENAI_AVAILABLE:
    import io, sys as _sys

    print('Running integration test with cot_n=2 (gpt-4o-mini)...')

    cot_config = AgentConfig(model='gpt-4o-mini', provider='openai')
    cot_config.lookup_sales_data.cot_n = 2
    cot_config.analyzing_data.cot_n    = 2

    cot_agent = SalesDataAgent(agent_config=cot_config)

    # Save Jupyter's stdout (NOT sys.__stdout__ which bypasses the kernel)
    jupyter_stdout = _sys.stdout
    captured = io.StringIO()
    _sys.stdout = captured

    try:
        # run() without save_results returns (result_dict, score_variance)
        cot_result, _ = cot_agent.run(
            'What were the total sales in November 2021?',
            no_vis=True,
        )
    finally:
        # Always restore Jupyter's stdout, even if run() raises
        _sys.stdout = jupyter_stdout

    log = captured.getvalue()
    print(log)

    cot_lines = [l for l in log.splitlines() if 'CoT iteration' in l or 'CoT early stop' in l]
    print(f'CoT log lines found: {len(cot_lines)}')
    for line in cot_lines:
        print(f'  {line}')

    assert len(cot_lines) > 0, 'Expected at least one CoT iteration log line'

    answer = cot_result.get('answer', [''])[0]
    sql    = cot_result.get('sql_query', '')
    assert answer, 'Expected a non-empty answer'
    assert sql,    'Expected a non-empty SQL query'

    print(f'\nFinal answer: {answer[:120]}')
    print(f'Final SQL:    {sql[:120]}')
    print('\nIntegration test with cot_n=2 passed ✓')

else:
    print('Skipping integration test (OPENAI_API_KEY not set)')


## Summary

| # | Test | What it checks | API needed |
|---|------|----------------|------------|
| 2.1 | `_extract_step_output` | Correct key extracted per step type | No |
| 2.2 | `CoTRefinementLLM` | Refinement suffix appended to every prompt | No |
| 2.3 | Early stop | Identical output halts iterations immediately | No |
| 2.4 | Full iterations | Dissimilar outputs run all cot_n passes | No |
| 2.5 | cot_n=1 no-op | No extra calls when refinement is disabled | No |
| 3.x | Config regression | `cot_n` serializes; existing fields unchanged | No |
| 4.x | Best-of-n regression | Zero overhead with default cot_n=1 | No |
| 5 | Integration | Real LLM produces CoT log lines | **Yes** |

## 6. Integration Test — Complex Multi-Step Query with cot_n=3

A deliberately hard question that requires CTEs / subqueries, half-year aggregations, percentage growth computation, and a secondary category breakdown — all in a single natural-language prompt.  
CoT iterative refinement has real room to improve the SQL here.

In [ ]:
if OPENAI_AVAILABLE:
    import io, sys as _sys

    COMPLEX_QUERY = (
        "Which sales representative achieved the highest revenue growth rate "
        "(in percentage) from the first half of 2021 (Jan–Jun) to the second half "
        "(Jul–Dec)? For that representative, also list their top 3 product categories "
        "ranked by total revenue in H2 2021, including the revenue figure for each."
    )

    print("Running complex integration test with cot_n=3 (gpt-4o-mini)...")
    print(f"Query: {COMPLEX_QUERY}\n")

    # Configure: apply cot_n=3 to the two steps that generate SQL / analysis
    complex_config = AgentConfig(model='gpt-4o-mini', provider='openai')
    complex_config.lookup_sales_data.cot_n = 3
    complex_config.analyzing_data.cot_n    = 3

    complex_agent = SalesDataAgent(agent_config=complex_config)

    jupyter_stdout = _sys.stdout
    captured = io.StringIO()
    _sys.stdout = captured

    try:
        complex_result, score_var = complex_agent.run(
            COMPLEX_QUERY,
            no_vis=True,
        )
    finally:
        _sys.stdout = jupyter_stdout

    log = captured.getvalue()
    print(log)

    # ── CoT trace ──────────────────────────────────────────────────────────────
    cot_lines = [l for l in log.splitlines()
                 if 'CoT iteration' in l or 'CoT early stop' in l]
    print(f"CoT log lines: {len(cot_lines)}")
    for line in cot_lines:
        print(f"  {line}")

    # ── Assertions ─────────────────────────────────────────────────────────────
    assert len(cot_lines) > 0, "Expected at least one CoT log line"

    sql = complex_result.get('sql_query', '')
    assert sql, "Expected a non-empty SQL query"

    # The query is complex enough that the LLM should use a CTE or subquery
    sql_upper = sql.upper()
    has_substructure = ('WITH ' in sql_upper or 'SELECT' in sql_upper.replace('SELECT', '', 1))
    print(f"\nSQL uses CTE / subquery: {has_substructure}")
    print(f"SQL length: {len(sql)} chars")
    print(f"\nFinal SQL:\n{sql}\n")

    answer_parts = complex_result.get('answer', [])
    answer_text  = answer_parts[-1] if answer_parts else ''
    assert answer_text, "Expected a non-empty answer"
    print(f"Final answer:\n{answer_text[:500]}")

    print(f"\nScore variance across CoT iterations: {score_var:.4f}")
    print("\nComplex integration test with cot_n=3 passed ✓")

else:
    print("Skipping complex integration test (OPENAI_API_KEY not set)")
